In [ ]:
import pandas as pd
import polars as pl
import seaborn as sns
import matplotlib.pyplot as plt
import os
from pathlib import Path
import polars.selectors as cs

In [ ]:
input_folder=Path().resolve().parent /'input'
input_folder

In [ ]:
pl.enable_string_cache()
pl.Config.set_streaming_chunk_size(2000000)

Building customer demographics behaviour

In [34]:
%%time
data=pl.scan_parquet(input_folder/'load_profile_buildingID_*')
BUILDINGS=data.collect().select(pl.col('bldg_id')).unique()

CPU times: user 1.01 s, sys: 38.7 ms, total: 1.05 s
Wall time: 161 ms


## The aim of this section is to find as many features as possible to build accurate hypothesis

In [ ]:
%%time
Meta_data=pl.scan_parquet(input_folder/'Meta_Data.parquet').with_columns(pl.col('bldg_id').cast(pl.UInt32)).filter(
    pl.col('bldg_id').is_in(BUILDINGS.to_series().to_list())).select(pl.col('in.occupants','in.state','in.county',
                        'in.representative_income','in.area_median_income','in.income',
                        'in.income_recs_2020','in.income_recs_2015', 'in.building_america_climate_zone',
                        'in.bedrooms','in.tenure','in.household_has_tribal_persons','bldg_id')
                ).unique()

In [35]:
data=data.join(Meta_data, on="bldg_id")
data.collect()

bldg_id,timestamp,in.sqft,out.electricity.ceiling_fan.energy_consumption..kwh,out.electricity.clothes_dryer.energy_consumption..kwh,out.electricity.clothes_washer.energy_consumption..kwh,out.electricity.cooling.energy_consumption..kwh,out.electricity.cooling_fans_pumps.energy_consumption..kwh,out.electricity.dishwasher.energy_consumption..kwh,out.electricity.ev_charging.energy_consumption..kwh,out.electricity.freezer.energy_consumption..kwh,out.electricity.heating.energy_consumption..kwh,out.electricity.heating_fans_pumps.energy_consumption..kwh,out.electricity.heating_hp_bkup.energy_consumption..kwh,out.electricity.heating_hp_bkup_fa.energy_consumption..kwh,out.electricity.hot_water.energy_consumption..kwh,out.electricity.hot_water_solar_th.energy_consumption..kwh,out.electricity.lighting_exterior.energy_consumption..kwh,out.electricity.lighting_garage.energy_consumption..kwh,out.electricity.lighting_interior.energy_consumption..kwh,out.electricity.mech_vent.energy_consumption..kwh,out.electricity.net.energy_consumption..kwh,out.electricity.permanent_spa_heat.energy_consumption..kwh,out.electricity.permanent_spa_pump.energy_consumption..kwh,out.electricity.plug_loads.energy_consumption..kwh,out.electricity.pool_heater.energy_consumption..kwh,out.electricity.pool_pump.energy_consumption..kwh,out.electricity.pv.energy_consumption..kwh,out.electricity.range_oven.energy_consumption..kwh,out.electricity.refrigerator.energy_consumption..kwh,out.electricity.television.energy_consumption..kwh,out.electricity.total.energy_consumption..kwh,out.electricity.well_pump.energy_consumption..kwh,out.fuel_oil.heating.energy_consumption..kwh,out.fuel_oil.hot_water.energy_consumption..kwh,out.fuel_oil.total.energy_consumption..kwh,out.natural_gas.clothes_dryer.energy_consumption..kwh,…,out.weather.direct_normal_solar_radiation..watt_per_m2,out.weather.wind_speed..meter_per_second,out.schedules.ceiling_fan,out.schedules.clothes_dryer,out.schedules.clothes_washer,out.schedules.cooking_range,out.schedules.cooling_setpoint..c,out.schedules.dishwasher,out.schedules.electric_vehicle_charging,out.schedules.electric_vehicle_discharging,out.schedules.heating_setpoint..c,out.schedules.hot_water_clothes_washer,out.schedules.hot_water_dishwasher,out.schedules.hot_water_fixtures,out.schedules.lighting_garage,out.schedules.lighting_interior,out.schedules.no_space_cooling,out.schedules.no_space_heating,out.schedules.occupants,out.schedules.peak_period,out.schedules.plug_loads_other,out.schedules.plug_loads_tv,out.schedules.power_outage,out.schedules.pre_peak_period,out.schedules.vacancy,in.occupants,in.state,in.county,in.representative_income,in.area_median_income,in.income,in.income_recs_2020,in.income_recs_2015,in.building_america_climate_zone,in.bedrooms,in.tenure,in.household_has_tribal_persons
u32,datetime[ms],f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str,str,str,f32,str,str,str,str,str,str,str,str
10,2018-01-01 00:15:00,1228.0,0.00514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.05603,0.00548,0.0,0.0,0.0,0.0,0.00088,0.0,0.00497,0.0,0.1662,0.0,0.0,0.05554,0.0,0.0,0.0,0.0,0.01194,0.01312,0.1662,0.01309,0.0,0.0,0.0,0.0,…,0.0,4.19993,0.566,0.0,0.0,0.0,24.7222,null,null,null,24.7222,0.0,null,0.0,null,0.0993,0.0,0.0,0.5,0.0,0.601,0.0699,0.0,0.0,0.0,"""4""","""FL""","""G1200860""",32414.0,"""30-60%""","""30000-34999""","""20000-39999""","""20000-39999""","""Hot-Humid""","""3""","""Owner""","""No"""
10,2018-01-01 00:30:00,1228.0,0.00514,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.04812,0.00467,0.0,0.0,0.0,0.0,0.00088,0.0,0.00477,0.0,0.15641,0.0,0.0,0.05554,0.0,0.0,0.0,0.0,0.01194,0.01225,0.15641,0.01309,0.0,0.0,0.0,0.0,…,0.0,3.300041,0.566,0.0,0.0,0.0,24.7222,null,null,null,24.7222,0.0,null,0.0,null,0.0952,0.0,0.0,0.5,0.0,0.601,0.0653,0.0,0.0,0.0,"""4""","""FL""","""G1200860""",32414.0,"""30-60

## First Hypothesis

the energy consumption patterns is significantly determined by the temporal attributes

## initial Data Preprocessing

In order to reach the desired objective we need to break the datetime into several columns each of which determines

In [ ]:
df_hp1=data.with_columns(
            pl.col('timestamp').dt.weekday().alias('day of the week'),
            pl.col('timestamp').dt.hour().alias('hour of the day'),
            pl.col('timestamp').dt.day().alias('day of the month'),
            pl.col('timestamp').dt.ordinal_day().alias('day of the year'),
            pl.col('timestamp').dt.week().alias('week of the year'),
            pl.col('timestamp').dt.month().alias('month of the year'),
            pl.col('out.electricity.cooling.energy_consumption..kwh').alias('out.electricity.AC.energy_consumption..kwh'),
            pl.col('timestamp').dt.quarter().alias('quarter')).with_columns(
            pl.when(pl.col('day of the week').is_in([6,7])).then(1).otherwise(0).alias('IsWeekend')
            ).select(cs.matches('^out.electricity.*|^bldg*|^day*|^hour*|^week*|^month*|^time*|^quarter|^IsWeekend')).collect()

* Checking missing values

In [ ]:
df_hp1.null_count()

No missing values within the data

In [ ]:
import importlib
import lib
from lib import loadProfile
importlib.reload(lib)
vis=loadProfile(df_hp1)

In [ ]:
# Run boxplots
vis.boxplot_exp()

In [ ]:
# Run histograms
vis.hist_exp()

In [ ]:
# Run Boxen plot for large distributions
vis.boxen_exp()

### Now we want to see the total amount spent if its holiday or not

In [ ]:
vis.barplot_seaborn('out.electricity.total.energy_consumption..kwh', 'IsWeekend')

In [ ]:
df_hp1.head(10)

## Visualization of the long duration appliances usage

Long duration appliances are water heaters, refrigerators, Gas Stove

### Objective 1: the hour of the day singificantly impact the consumption pattern

In [ ]:
long_duration=df_hp1.select('hour of the day', 
                            'out.electricity.heating.energy_consumption..kwh',
                           'out.electricity.cooling_fans_pumps.energy_consumption..kwh',
                           'out.electricity.AC.energy_consumption..kwh',
                            'out.electricity.hot_water.energy_consumption..kwh',
                            'out.electricity.refrigerator.energy_consumption..kwh',
                            'out.electricity.television.energy_consumption..kwh',
                            'out.electricity.total.energy_consumption..kwh',
                           ).pipe(vis.edit_column_names).group_by('hour of the day').agg(pl.all().sum()).sort('hour of the day')

In [ ]:
vis.display_totals(long_duration,'out.electricity.total', 'hour of the day')

In [ ]:
# apply correlation test to see which appliance contrinbutes more to the total consumption of energy
vis.test_corr(long_duration, 'out.electricity.total')

In [ ]:
vis.display_totals(long_duration,'out.electricity.AC', 'hour of the day')

## we Found that the ac correlates more to the values to the total consumption of energy

In [ ]:
vis.long_dev_hourly(long_duration)

In [ ]:
## now lets test if there is a correlation between the hour of the day and the total consunmption and some appliances
vis.test_corr(long_duration, 'hour of the day')

## the p-value for the correlation between all appliances and total consumption with the hour of the day is all less then 0.05
## Indicating a strong correlation, therefore we reject the null hypothesis and we accept the alternative
## therefore, hour of the day significantly determines the consumption patterns

### Objective 2: the day of the month singificantly impact the consumption pattern

### Objective 3: the week of the month singificantly impact the consumption pattern

### Objective 4: the month of the year singificantly impact the consumption pattern

## Second Hypothesis

net total energy consumption determines the peak load periods

## Third Hypothesis

Simultaneous operation of two appliance creates a load on user's side

In [ ]:
##TODO: Make a pairplot for all long duration appliances generally
## TODO: Make a pairplot for all long duration appliances using correlations like is_Holiday
sns.pairplot(data=long_duration.to_pandas())

This proves hypothesis four that there is a relationship between devices

## Fourth Hypothesis

User Behaviour analysis